# Table A — Step 3+4: VGGT Signals + Pose Scoring (New Pairs Only)

Runs `vggt_signals.py` and `pose_scoring.py` on the geometry filter outputs from Step 2,
**restricted to `new_pair=True` candidates only** — pairs not seen in the original DINO pipeline.
Pairs with `new_pair=False` already have DINO pipeline results and need no re-processing.

**Before running**, upload to Drive at `[DRIVE_ROOT]/scripts/`:
```
scripts/
    vggt_signals.py
    pose_scoring.py
```
`vggt_omega` is installed via pip in Step 4a — fill in your git URL there.

**Inputs** (produced by `geometry_filter_ablation_colab.ipynb`):
```
ablation_outputs/geometry_filter/
    sscd_shard1/
        vggt_candidates_manifest.jsonl
        aspan_all_manifest.jsonl
        aspan_sidecars/
    clip_shard1/  (same structure)
```

**Outputs** (written to Drive):
```
ablation_outputs/vggt/
    sscd_shard1/
        vggt_judged_manifest.jsonl
        pose_scored_manifest.jsonl
        pose_scoring_summary.json
    clip_shard1/  (same structure)
```

In [ ]:
# Step 0 — Fix cv2/numpy ABI mismatch and verify GPU.
# Colab ships an opencv wheel compiled against numpy 1.x. PyTorch 2.11+ pins numpy 2.x,
# so downgrading numpy is not an option. Upgrade opencv to 4.9+ which added numpy 2.x ABI
# support, then restart the runtime so the new wheel is active.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "opencv-python-headless>=4.9.0", "-q"],
    check=True,
)

import torch
import cv2
import numpy as np
print(f"PyTorch : {torch.__version__}")
print(f"numpy   : {np.__version__}")
print(f"cv2     : {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — switch runtime to GPU before continuing.")

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Configure paths
# ── EDIT THESE ──────────────────────────────────────────────────────────────
from pathlib import Path
import json, shutil, sys, os

DRIVE_ROOT  = Path('/content/drive/MyDrive/ImageSimilarity')
SCRIPTS_DIR = DRIVE_ROOT / 'scripts'

# Image corpus — same zips as geometry_filter_ablation_colab.ipynb
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'
LOCAL_SOURCE_DIR = Path('./source')
LOCAL_TARGET_DIR = Path('./target')

# VGGT-Omega checkpoint
VGGT_CHECKPOINT = DRIVE_ROOT / 'weights' / 'vggt_omega_1b_512.pt'  # adjust filename if needed

# Geometry filter outputs on Drive (inputs to this notebook)
GF_OUT       = DRIVE_ROOT / 'ablation_outputs' / 'geometry_filter'
SSCD_GF_DIR  = GF_OUT / 'sscd_shard1'
CLIP_GF_DIR  = GF_OUT / 'clip_shard1'
# ────────────────────────────────────────────────────────────────────────────

LOCAL_WORK = Path('/content/vggt_ablation')
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

DRIVE_OUT = DRIVE_ROOT / 'ablation_outputs' / 'vggt'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print(f"VGGT checkpoint : {VGGT_CHECKPOINT}  (exists: {VGGT_CHECKPOINT.exists()})")
print(f"SSCD GF dir     : {SSCD_GF_DIR}  (exists: {SSCD_GF_DIR.exists()})")
print(f"CLIP GF dir     : {CLIP_GF_DIR}  (exists: {CLIP_GF_DIR.exists()})")
print(f"Source zip      : {SOURCE_ZIP}  (exists: {SOURCE_ZIP.exists()})")
print(f"Target zip      : {TARGET_ZIP}  (exists: {TARGET_ZIP.exists()})")

In [ ]:
# Step 3 — Pre-run statistics
# Reads manifests directly from Drive — no GPU, no model loading.
# Shows exactly how many new_pair=True candidates will be sent to VGGT,
# and warns if there is nothing to do before any expensive work starts.

def pre_stats(all_manifest_path, vggt_manifest_path, label):
    print(f"── {label} {'─' * 42}")

    # aspan_all_manifest: full geometry filter picture
    if all_manifest_path.exists():
        n_total = n_cached = n_pass = n_new_total = n_new_pass = 0
        with open(all_manifest_path, encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                row = json.loads(line)
                n_total += 1
                if row.get('from_cache'): n_cached += 1
                if row.get('aspan_pass'): n_pass += 1
                if row.get('new_pair'):   n_new_total += 1
                if row.get('new_pair') and row.get('aspan_pass'): n_new_pass += 1
        print(f"  Geometry filter: {n_total:,} pairs attempted")
        print(f"    Cached (ASpanFormer skipped) : {n_cached:,}  ({100*n_cached/max(n_total,1):.1f}%)")
        print(f"    Passed keypoint filter       : {n_pass:,}")
        print(f"    New pairs (not in DINO)      : {n_new_total:,} total,  {n_new_pass:,} passed filter")
    else:
        print(f"  aspan_all_manifest not found: {all_manifest_path}")

    # vggt_candidates_manifest: what VGGT would receive without filtering
    n_new = 0
    if vggt_manifest_path.exists():
        n_vggt = n_new = n_old = 0
        with open(vggt_manifest_path, encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                row = json.loads(line)
                n_vggt += 1
                if row.get('new_pair'): n_new += 1
                else: n_old += 1
        print(f"  VGGT candidates: {n_vggt:,} pairs passed ASpanFormer")
        print(f"    new_pair=True  : {n_new:,}  ← VGGT will run on these")
        print(f"    new_pair=False : {n_old:,}  ← already in DINO pipeline, skipped")
        if n_new == 0:
            print(f"    WARNING: 0 new pairs — nothing to run for {label}")
    else:
        print(f"  vggt_candidates_manifest not found: {vggt_manifest_path}")
    return n_new

print("=" * 60)
print("Pre-run manifest statistics (reading from Drive, no GPU)")
print("=" * 60)
print()
N_SSCD_NEW = pre_stats(
    SSCD_GF_DIR / 'aspan_all_manifest.jsonl',
    SSCD_GF_DIR / 'vggt_candidates_manifest.jsonl',
    'SSCD Shard 1',
)
print()
N_CLIP_NEW = pre_stats(
    CLIP_GF_DIR / 'aspan_all_manifest.jsonl',
    CLIP_GF_DIR / 'vggt_candidates_manifest.jsonl',
    'CLIP Shard 1',
)
print()
print(f"Total new pairs to process: {N_SSCD_NEW + N_CLIP_NEW}  "
      f"(SSCD: {N_SSCD_NEW}, CLIP: {N_CLIP_NEW})")
if N_SSCD_NEW + N_CLIP_NEW == 0:
    print("\nNothing to do — all candidates were already seen in the DINO pipeline.")

In [ ]:
# Step 3b — Coverage gap: DINO-approved candidates missed entirely by SSCD / CLIP retrieval.
# "DINO-approved" = aspan_pass=True in the DINO cache manifests — pairs that passed the
# keypoint filter and went on to VGGT+pose scoring. Precision at this stage is very high,
# so this is a good proxy for verified true matches without needing the labeled CSVs.
# A pair in the gap was never even retrieved by SSCD/CLIP, so no downstream filtering
# could recover it — it is an irreducible retrieval miss.

DINO_SHARD1_CACHE = SCRIPTS_DIR / '_local' / 'aspan_all_manifest_shard1_reconstructed.jsonl'

def load_dino_approved(cache_path):
    approved = set()
    if not Path(str(cache_path)).exists():
        print(f"  Not found: {cache_path}")
        return approved
    with open(str(cache_path), encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            if row.get('aspan_pass'):
                approved.add((str(row['source_id']), str(row['target_id'])))
    return approved

def load_all_retrieved(aspan_all_path):
    pairs = set()
    if not Path(str(aspan_all_path)).exists():
        print(f"  Not found: {aspan_all_path}")
        return pairs
    with open(str(aspan_all_path), encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            pairs.add((str(row['source_id']), str(row['target_id'])))
    return pairs

print("=" * 60)
print("Coverage gap: DINO-approved pairs missed by SSCD / CLIP")
print("=" * 60)
print()

dino_s1 = load_dino_approved(DINO_SHARD1_CACHE)
print(f"DINO Shard 1 — ASpanFormer-approved pairs : {len(dino_s1):,}")

sscd_s1_all = load_all_retrieved(SSCD_GF_DIR / 'aspan_all_manifest.jsonl')
clip_s1_all = load_all_retrieved(CLIP_GF_DIR / 'aspan_all_manifest.jsonl')
print(f"SSCD Shard 1 — all retrieved candidates   : {len(sscd_s1_all):,}")
print(f"CLIP Shard 1 — all retrieved candidates   : {len(clip_s1_all):,}")
print()

sscd_hit  = dino_s1 & sscd_s1_all
clip_hit  = dino_s1 & clip_s1_all
sscd_miss = dino_s1 - sscd_s1_all
clip_miss = dino_s1 - clip_s1_all
either_hit = dino_s1 & (sscd_s1_all | clip_s1_all)
both_miss  = dino_s1 - sscd_s1_all - clip_s1_all

n = max(len(dino_s1), 1)
print(f"── SSCD {'─' * 52}")
print(f"  Retrieved  : {len(sscd_hit):,}  ({100*len(sscd_hit)/n:.1f}% of DINO-approved)")
print(f"  Missed     : {len(sscd_miss):,}  ({100*len(sscd_miss)/n:.1f}%)  ← retrieval gap")
print()
print(f"── CLIP {'─' * 52}")
print(f"  Retrieved  : {len(clip_hit):,}  ({100*len(clip_hit)/n:.1f}% of DINO-approved)")
print(f"  Missed     : {len(clip_miss):,}  ({100*len(clip_miss)/n:.1f}%)  ← retrieval gap")
print()
print(f"── SSCD ∪ CLIP combined {'─' * 36}")
print(f"  Covered by either  : {len(either_hit):,}  ({100*len(either_hit)/n:.1f}%)")
print(f"  Missed by BOTH     : {len(both_miss):,}  ({100*len(both_miss)/n:.1f}%)  ← absolute gap")
print()
if both_miss:
    print("  Sample pairs missed by both (first 5):")
    for src, tgt in list(both_miss)[:5]:
        print(f"    source: {src}")
        print(f"    target: {tgt}")
        print()

In [ ]:
# Step 4a — Install vggt_omega via pip
# TODO: replace the URL below with the actual git URL you used previously
# Example: !pip install git+https://github.com/org/vggt-omega.git -q
VGGT_GIT_URL = "git+https://github.com/TODO/vggt-omega.git"  # ← EDIT THIS

import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", VGGT_GIT_URL, "-q"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"pip install failed for {VGGT_GIT_URL}")

# Verify import
from vggt_omega.models import VGGTOmega
print("vggt_omega imported successfully.")

In [ ]:
# Step 4b — Copy pipeline scripts from Drive
for src, dst in [
    (SCRIPTS_DIR / 'vggt_signals.py', LOCAL_WORK / 'vggt_signals.py'),
    (SCRIPTS_DIR / 'pose_scoring.py', LOCAL_WORK / 'pose_scoring.py'),
]:
    if not Path(str(src)).exists():
        raise FileNotFoundError(f"Missing on Drive: {src}")
    shutil.copy(str(src), str(dst))
    print(f"Copied: {Path(src).name}")

if str(LOCAL_WORK) not in sys.path:
    sys.path.insert(0, str(LOCAL_WORK))
print("Scripts ready.")

In [ ]:
# Step 4c — Extract and flatten image corpus
# vggt_signals.py reads source and target images to reconstruct the ASpan alignment.
import time

def _extract_and_flatten(zip_src, local_name):
    dst_dir = Path(f'./{local_name}')
    if dst_dir.exists() and any(dst_dir.iterdir()):
        n = sum(1 for p in dst_dir.iterdir() if p.is_file())
        print(f"  {local_name}/: already present ({n:,} files)")
        return n
    local_zip = Path(f'./{local_name}.zip')
    print(f"  Copying {Path(str(zip_src)).name} from Drive ...")
    shutil.copy(str(zip_src), str(local_zip))
    t0 = time.time()
    os.system(f'unzip -q {local_zip} -d {local_name}')
    print(f"  Unzipped in {time.time()-t0:.0f}s")
    for root, dirs, files in os.walk(str(dst_dir)):
        if root == str(dst_dir): continue
        for fname in files:
            src_path = os.path.join(root, fname)
            dst_path = os.path.join(str(dst_dir), fname)
            if os.path.exists(dst_path):
                base, ext = os.path.splitext(fname)
                dst_path = os.path.join(str(dst_dir), f"{base}_{root.replace('/', '_')}{ext}")
            shutil.move(src_path, dst_path)
    for root, dirs, files in os.walk(str(dst_dir), topdown=False):
        if root == str(dst_dir): continue
        try: os.rmdir(root)
        except OSError: pass
    local_zip.unlink(missing_ok=True)
    n = sum(1 for p in dst_dir.iterdir() if p.is_file())
    print(f"  {local_name}/: {n:,} files ready")
    return n

_extract_and_flatten(SOURCE_ZIP, 'source')
_extract_and_flatten(TARGET_ZIP, 'target')

In [ ]:
# Step 4d — Copy geometry filter manifests and sidecars from Drive to local
# Sidecars are NPZ files vggt_signals.py uses to reconstruct the ASpan alignment.

def copy_gf_run(gf_drive_dir, local_dir, label):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    src_m = Path(str(gf_drive_dir)) / 'vggt_candidates_manifest.jsonl'
    dst_m = local_dir / 'vggt_candidates_manifest.jsonl'
    shutil.copy(str(src_m), str(dst_m))
    print(f"  {label}: manifest copied ({src_m.stat().st_size / 1e6:.1f} MB)")

    src_s = Path(str(gf_drive_dir)) / 'aspan_sidecars'
    dst_s = local_dir / 'aspan_sidecars'
    if src_s.exists():
        if dst_s.exists():
            shutil.rmtree(str(dst_s))
        shutil.copytree(str(src_s), str(dst_s))
        n = len(list(dst_s.glob('*.npz')))
        print(f"  {label}: {n} sidecars copied")
    else:
        print(f"  {label}: WARNING — no aspan_sidecars directory found at {src_s}")
    return dst_m, dst_s

SSCD_LOCAL_MANIFEST, SSCD_LOCAL_SIDECARS = copy_gf_run(SSCD_GF_DIR, LOCAL_WORK / 'sscd_shard1', 'SSCD shard1')
CLIP_LOCAL_MANIFEST, CLIP_LOCAL_SIDECARS = copy_gf_run(CLIP_GF_DIR, LOCAL_WORK / 'clip_shard1', 'CLIP shard1')

In [ ]:
# Step 5 — Filter to new_pair=True and rewrite all paths to this session's local filesystem.
# source_path/target_path were absolute paths from the geometry filter session — rewrite
# to the local image dirs extracted in Step 4c.
# sidecar_path was an absolute path in the geometry filter session — rewrite to the
# local sidecars dir copied in Step 4d.

def filter_new_pairs(src_manifest, dst_manifest, sidecar_local_dir, label):
    rows = []
    n_total = 0
    with open(str(src_manifest), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            n_total += 1
            if not row.get('new_pair'):
                continue
            if 'source_path' in row:
                row['source_path'] = str(LOCAL_SOURCE_DIR.resolve() / Path(row['source_path']).name)
            if 'target_path' in row:
                row['target_path'] = str(LOCAL_TARGET_DIR.resolve() / Path(row['target_path']).name)
            if 'sidecar_path' in row:
                row['sidecar_path'] = str(Path(sidecar_local_dir) / Path(row['sidecar_path']).name)
            rows.append(json.dumps(row, ensure_ascii=False))

    Path(str(dst_manifest)).parent.mkdir(parents=True, exist_ok=True)
    Path(str(dst_manifest)).write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
    print(f"  {label}: {len(rows)}/{n_total} rows kept (new_pair=True)")

    # Spot-check: verify first row's paths exist
    if rows:
        r = json.loads(rows[0])
        print(f"    sample src    : {r.get('source_path')}  (exists: {Path(str(r.get('source_path',''))).exists()})")
        print(f"    sample tgt    : {r.get('target_path')}  (exists: {Path(str(r.get('target_path',''))).exists()})")
        print(f"    sample sidecar: {r.get('sidecar_path')}  (exists: {Path(str(r.get('sidecar_path',''))).exists()})")
    return len(rows)

SSCD_NEW_MANIFEST = LOCAL_WORK / 'sscd_shard1' / 'new_pairs_manifest.jsonl'
CLIP_NEW_MANIFEST = LOCAL_WORK / 'clip_shard1' / 'new_pairs_manifest.jsonl'

n_sscd_new = filter_new_pairs(SSCD_LOCAL_MANIFEST, SSCD_NEW_MANIFEST, SSCD_LOCAL_SIDECARS, 'SSCD shard1')
n_clip_new = filter_new_pairs(CLIP_LOCAL_MANIFEST, CLIP_NEW_MANIFEST, CLIP_LOCAL_SIDECARS, 'CLIP shard1')
print(f"\nTotal new pairs to submit to VGGT: {n_sscd_new + n_clip_new}")

## SSCD Shard 1 — VGGT + Pose Scoring

In [ ]:
import importlib, vggt_signals
importlib.reload(vggt_signals)

SSCD_VGGT_OUT = LOCAL_WORK / 'sscd_shard1' / 'vggt_output'

if n_sscd_new == 0:
    print("No new SSCD pairs — skipping VGGT.")
else:
    vggt_signals.main([
        '--input-manifest', str(SSCD_NEW_MANIFEST),
        '--output-dir',     str(SSCD_VGGT_OUT),
        '--checkpoint',     str(VGGT_CHECKPOINT),
        '--device',         'auto',
        '--progress-every', '5',
        '--resume',
    ])

In [ ]:
import pose_scoring
importlib.reload(pose_scoring)

SSCD_POSE_OUT = LOCAL_WORK / 'sscd_shard1' / 'pose_output'

if n_sscd_new == 0:
    print("No new SSCD pairs — skipping pose scoring.")
else:
    pose_scoring.main([
        '--input-manifest', str(SSCD_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(SSCD_POSE_OUT),
    ])

## CLIP Shard 1 — VGGT + Pose Scoring

In [ ]:
importlib.reload(vggt_signals)

CLIP_VGGT_OUT = LOCAL_WORK / 'clip_shard1' / 'vggt_output'

if n_clip_new == 0:
    print("No new CLIP pairs — skipping VGGT.")
else:
    vggt_signals.main([
        '--input-manifest', str(CLIP_NEW_MANIFEST),
        '--output-dir',     str(CLIP_VGGT_OUT),
        '--checkpoint',     str(VGGT_CHECKPOINT),
        '--device',         'auto',
        '--progress-every', '5',
        '--resume',
    ])

In [ ]:
importlib.reload(pose_scoring)

CLIP_POSE_OUT = LOCAL_WORK / 'clip_shard1' / 'pose_output'

if n_clip_new == 0:
    print("No new CLIP pairs — skipping pose scoring.")
else:
    pose_scoring.main([
        '--input-manifest', str(CLIP_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(CLIP_POSE_OUT),
    ])

In [ ]:
# Post-run statistics — how many new pairs survived the final decision gate.
# Gate: aspan_2d_inlier_ratio >= 0.65  AND  pose_component_score <= 2.13

def post_stats(pose_out_dir, label, n_new_input):
    print(f"── {label} {'─' * 42}")
    summary_path = Path(str(pose_out_dir)) / 'pose_scoring_summary.json'
    if not summary_path.exists():
        print(f"  No results yet (pose_scoring_summary.json not found)")
        return 0
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    n_total    = summary['n_total']
    n_accepted = summary['n_accepted']
    n_rejected = summary['n_rejected']
    pct = 100 * n_accepted / n_total if n_total > 0 else 0.0
    print(f"  New pairs submitted     : {n_new_input}")
    print(f"  Passed final gate       : {n_accepted}/{n_total}  ({pct:.1f}%)")
    print(f"  Rejected                : {n_rejected}")
    print(f"  Rejection breakdown:")
    for reason, count in sorted(summary['reason_counts'].items(), key=lambda kv: -kv[1]):
        if reason != 'pass':
            pct_r = 100 * count / n_total if n_total > 0 else 0.0
            print(f"    {reason}: {count}  ({pct_r:.1f}%)")
    return n_accepted

print("=" * 60)
print("Post-run: new pairs through the final decision gate")
print("  (inlier_ratio >= 0.65  AND  pose_component_score <= 2.13)")
print("=" * 60)
print()
n_sscd_accepted = post_stats(SSCD_POSE_OUT, 'SSCD Shard 1', n_sscd_new)
print()
n_clip_accepted = post_stats(CLIP_POSE_OUT, 'CLIP Shard 1', n_clip_new)
print()
print(f"Total new true matches found: {n_sscd_accepted + n_clip_accepted}  "
      f"(SSCD: {n_sscd_accepted}, CLIP: {n_clip_accepted})")

In [ ]:
# Copy outputs to Drive
runs = [
    ('sscd_shard1', SSCD_VGGT_OUT, SSCD_POSE_OUT),
    ('clip_shard1', CLIP_VGGT_OUT, CLIP_POSE_OUT),
]

for tag, vggt_out, pose_out in runs:
    dst_dir = DRIVE_OUT / tag
    dst_dir.mkdir(parents=True, exist_ok=True)
    files_to_save = [
        (Path(str(vggt_out)) / 'vggt_judged_manifest.jsonl',  dst_dir / 'vggt_judged_manifest.jsonl'),
        (Path(str(vggt_out)) / 'true_matches_manifest.jsonl', dst_dir / 'true_matches_manifest.jsonl'),
        (Path(str(vggt_out)) / 'vggt_judge_summary.json',     dst_dir / 'vggt_judge_summary.json'),
        (Path(str(pose_out)) / 'pose_scored_manifest.jsonl',  dst_dir / 'pose_scored_manifest.jsonl'),
        (Path(str(pose_out)) / 'pose_scoring_summary.json',   dst_dir / 'pose_scoring_summary.json'),
    ]
    for src, dst in files_to_save:
        if src.exists():
            shutil.copy(str(src), str(dst))
            print(f"  Saved: {tag}/{dst.name}  ({src.stat().st_size / 1e6:.2f} MB)")
        else:
            print(f"  MISSING: {tag}/{dst.name}")

print(f"\nAll outputs at: {DRIVE_OUT}")